# Feature engineering — per-window MCSA + time-domain features

**Input:** `window_index.csv`.

**Output:** `data/processed/features_window.csv` — one row per window of BOTH
phase currents (36 features, no vibration).

Time-domain: `rms, std, peak, crest, skew, kurt` (mean dropped — DC offset).
Spectral (Welch PSD, 3.9 Hz bins): sideband ratios at LINE_FREQ ± k·fault_hz and
fault-energy share, for BPFO/BPFI/BSF/FTF.

Run once (~minutes; re-runs just re-read nothing here — it regenerates).

In [1]:
# Extract per-window features for the whole corpus (MCSA-only)
import pandas as pd
from utils import build_metadata
from pipeline.features import build_feature_set

from pipeline.config import PROCESSED_DIR
index = pd.read_csv(PROCESSED_DIR / "window_index.csv")
feats = build_feature_set(build_metadata(refresh=False), index, save=True)
print(f"rows x cols: {feats.shape}")
display(feats.head())

rows x cols: (76868, 42)


,recording,bearing,condition,rep,label,window_id,pc1_rms,pc1_std,pc1_peak,pc1_crest,...,pc2_er_bpfo,pc2_sb_bpfi_lo,pc2_sb_bpfi_hi,pc2_er_bpfi,pc2_sb_bsf_lo,pc2_sb_bsf_hi,pc2_er_bsf,pc2_sb_ftf_lo,pc2_sb_ftf_hi,pc2_er_ftf
0,N09_M07_F10_K001_1.mat,K001,N09_M07_F10,1,healthy,0,1.655065,1.654359,2.674539,1.615973,...,0.000516,0.117419,36.932583,0.007212,0.770577,157.539964,0.000011,79.365187,443020.958912,0.000665
1,N09_M07_F10_K001_1.mat,K001,N09_M07_F10,1,healthy,1,1.662702,1.662536,2.675917,1.609378,...,0.000575,1.297016,25.100337,0.007398,3.785049,93.702437,0.000010,25.885661,321471.238268,0.000055
2,N09_M07_F10_K001_1.mat,K001,N09_M07_F10,1,healthy,2,1.651330,1.651066,2.730331,1.653413,...,0.000595,1.460672,57.124265,0.007304,10.519177,73.862998,0.000021,192.830052,498735.839477,0.000381
3,N09_M07_F10_K001_1.mat,K001,N09_M07_F10,1,healthy,3,1.658319,1.657384,2.730331,1.646445,...,0.000484,2.426317,73.630223,0.007211,10.724982,92.773173,0.000011,218.797854,641831.060036,0.000624
4,N09_M07_F10_K001_1.mat,K001,N09_M07_F10,1,healthy,4,1.659412,1.659311,2.816429,1.697245,...,0.000524,22.407416,42.383150,0.007239,16.619998,41.283088,0.000013,288.379407,519267.553725,0.000002


In [2]:
# Feature layout + integrity
feature_cols = [c for c in feats.columns if c.startswith("pc1_") or c.startswith("pc2_")]
print(f"feature columns: {len(feature_cols)}")
print("NaNs:", int(feats.isna().sum().sum()))
print("\nlabel balance (windows):")
print(feats["label"].value_counts().to_string())
print("\nmean of key features by label:")
display(feats.groupby("label")[["pc1_rms", "pc1_kurt", "pc1_sb_bpfo_hi", "pc1_er_bpfo"]].mean().round(4))

feature columns: 36
NaNs: 0

label balance (windows):
label
damaged    62458
healthy    14410

mean of key features by label:


,pc1_rms,pc1_kurt,pc1_sb_bpfo_hi,pc1_er_bpfo
label,,,,
damaged,1.5259,-1.4943,77.1957,0.0002
healthy,1.5286,-1.4939,84.9629,0.0002


## Findings
- **76,868 windows × 42** (36 features + keys), **zero NaNs**.
- Class split at window level = 62,458 damaged : 14,410 healthy (~4.3 : 1).
- Broadband time-domain features barely separate classes (per exploration);
  the MCSA sideband/energy features carry the discrimination signal.